# 02 — Preparação dos Dados

Esta seção documenta a preparação dos dados utilizados no experimento.

O experimento utiliza um recorte derivado da lógica do Open-Prompt-Injection, combinando:

- uma tarefa legítima de classificação de sentimento;
- uma tarefa adversarial injetada de classificação de spam;
- versões limpas e versões contaminadas por prompt injection;
- separação entre ataques vistos e não vistos durante o treinamento.

A tarefa legítima é derivada do SST-2, enquanto a tarefa injetada é derivada do SMS Spam Collection. A combinação cria exemplos nos quais o modelo deve executar a tarefa original mesmo quando uma instrução adversarial aparece no contexto.

Os dados são organizados em dois níveis:

1. **Formato canônico**, usado como representação comum do experimento;
2. **Views de treino e avaliação**, usadas por cada cenário experimental.

## 1. Parâmetros da preparação dos dados

Nesta execução, o dataset foi preparado com os seguintes parâmetros:

- `n_train = 5000`;
- `n_test = 872`;
- `dataset_seed = 42`.

A seed do dataset controla o pareamento entre exemplos da tarefa legítima e exemplos da tarefa injetada, além da organização das amostras. Essa seed é mantida fixa para todos os cenários, garantindo que C0, C1, C2, C3 e C4 sejam avaliados sobre os mesmos dados.

As seeds de treinamento são tratadas separadamente e serão utilizadas apenas nos cenários com ajuste do modelo: C2, C3 e C4.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import shutil
import subprocess
import sys

import pandas as pd
import numpy as np

# Diretório principal do projeto
PROJECT_DIR = Path("/workspace/pi-defense-exp")

# Diretórios globais do projeto
DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
DATA_RUNS_DIR = DATA_DIR / "runs"

SCRIPTS_DIR = PROJECT_DIR / "scripts"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
CACHE_DIR = PROJECT_DIR / "cache"
EXPORTS_DIR = PROJECT_DIR / "exports"

# Diretórios globais legados/ativos
# Estes diretórios continuam úteis como área ativa de trabalho.
LOGS_DIR = PROJECT_DIR / "logs"
RESULTS_DIR = PROJECT_DIR / "results"
RAW_RESULTS_DIR = RESULTS_DIR / "raw"
METRICS_DIR = RESULTS_DIR / "metrics"
GENERATIONS_DIR = RESULTS_DIR / "generations"
ENVIRONMENT_DIR = RESULTS_DIR / "environment"

# Configuração principal do experimento
EXPERIMENT_NAME = "opi_sst2_sms_5k_872"

N_TRAIN = 5000
N_TEST = 872

# Seed usada para construir o dataset.
# Ela deve permanecer fixa para todos os cenários.
DATASET_SEED = 42

# Seeds usadas nos treinamentos dos cenários C2, C3 e C4.
TRAINING_SEEDS = [42, 123, 2026]

# Estrutura versionada da execução
RUN_ROOT = PROJECT_DIR / "runs" / EXPERIMENT_NAME / f"dataset_seed{DATASET_SEED}"
RUN_DATA_DIR = RUN_ROOT / "data"

BASELINE_ROOT = RUN_ROOT / "baselines" / "c0_c1"

# Diretórios que devem existir antes da preparação dos dados
required_directories = [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    DATA_RUNS_DIR,
    SCRIPTS_DIR,
    NOTEBOOKS_DIR,
    CACHE_DIR,
    EXPORTS_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    RAW_RESULTS_DIR,
    METRICS_DIR,
    GENERATIONS_DIR,
    ENVIRONMENT_DIR,
    RUN_ROOT,
    RUN_DATA_DIR,
    BASELINE_ROOT / "logs",
    BASELINE_ROOT / "raw",
    BASELINE_ROOT / "metrics",
]

for seed in TRAINING_SEEDS:
    seed_root = RUN_ROOT / f"train_seed{seed}"
    required_directories.extend([
        seed_root / "adapters" / "c2_struq_sft",
        seed_root / "adapters" / "c3_secalign_dpo",
        seed_root / "adapters" / "c4_instruction_hierarchy_sft",
        seed_root / "logs",
        seed_root / "raw",
        seed_root / "metrics",
    ])

for directory in required_directories:
    directory.mkdir(parents=True, exist_ok=True)

experiment_config = {
    "project_dir": str(PROJECT_DIR),
    "experiment_name": EXPERIMENT_NAME,
    "n_train": N_TRAIN,
    "n_test": N_TEST,
    "dataset_seed": DATASET_SEED,
    "training_seeds": TRAINING_SEEDS,
    "run_root": str(RUN_ROOT),
    "run_data_dir": str(RUN_DATA_DIR),
    "baseline_root": str(BASELINE_ROOT),
}

pd.DataFrame([experiment_config])

In [ ]:
directory_check_df = pd.DataFrame([
    {
        "directory": str(directory.relative_to(PROJECT_DIR)),
        "exists": directory.exists(),
        "path": str(directory),
    }
    for directory in required_directories
])

directory_check_df

## 2. Scripts utilizados na preparação dos dados

A preparação dos dados usa dois scripts principais:

1. `build_canonical_from_opi_sst2_sms.py`

   Responsável por gerar os arquivos canônicos:

   - `canonical_train.jsonl`;
   - `canonical_test.jsonl`.

2. `build_training_views.py`

   Responsável por gerar as views específicas dos cenários:

   - `train_struq_sft.jsonl`;
   - `train_secalign_dpo.jsonl`;
   - `train_ih_sft.jsonl`;
   - `test_all_scenarios.jsonl`.

O arquivo canônico representa os dados em um formato comum. As views transformam esse formato comum em entradas adequadas para cada cenário experimental.

In [ ]:
required_data_scripts = {
    "build_canonical_from_opi_sst2_sms": SCRIPTS_DIR / "build_canonical_from_opi_sst2_sms.py",
    "build_training_views": SCRIPTS_DIR / "build_training_views.py",
}

script_checks = []

for name, path in required_data_scripts.items():
    script_checks.append({
        "script": name,
        "path": str(path),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
    })

pd.DataFrame(script_checks)

## 3. Geração do dataset canônico

O dataset canônico é gerado a partir da combinação entre SST-2 e SMS Spam.

Comando utilizado:

```bash
python scripts/build_canonical_from_opi_sst2_sms.py \
  --n_train 5000 \
  --n_test 872 \
  --seed 42 \
  2>&1 | tee logs/01_build_canonical_from_opi_sst2_sms_5k_872.log
```

In [ ]:
RUN_DATA_PREP = False

canonical_log_path = LOGS_DIR / "01_build_canonical_from_opi_sst2_sms_5k_872.log"

canonical_command = [
    sys.executable,
    str(SCRIPTS_DIR / "build_canonical_from_opi_sst2_sms.py"),
    "--n_train", str(N_TRAIN),
    "--n_test", str(N_TEST),
    "--seed", str(DATASET_SEED),
]

print("Comando de geração canônica:")
print(" ".join(canonical_command))

if RUN_DATA_PREP:
    LOGS_DIR.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        canonical_command,
        capture_output=True,
        text=True,
    )

    canonical_log_path.write_text(
        result.stdout + "\n" + result.stderr,
        encoding="utf-8"
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Erro ao gerar dataset canônico.")

    print("Log salvo em:", canonical_log_path)
else:
    print("RUN_DATA_PREP=False. A geração não foi executada nesta célula.")

## 4. Arquivos canônicos esperados

Após a geração do dataset canônico, os seguintes arquivos devem existir em `data/processed`:

- `canonical_train.jsonl`;
- `canonical_test.jsonl`.

O arquivo `canonical_train.jsonl` contém as amostras usadas para gerar os dados de treinamento dos cenários C2, C3 e C4.

O arquivo `canonical_test.jsonl` contém as amostras usadas para avaliação final, incluindo ataques vistos e não vistos.

In [ ]:
canonical_files = {
    "canonical_train": PROCESSED_DATA_DIR / "canonical_train.jsonl",
    "canonical_test": PROCESSED_DATA_DIR / "canonical_test.jsonl",
}

canonical_file_checks = []

for name, path in canonical_files.items():
    canonical_file_checks.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else None,
    })

pd.DataFrame(canonical_file_checks)

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0

    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


canonical_counts = []

for name, path in canonical_files.items():
    canonical_counts.append({
        "file": name,
        "lines": count_jsonl_lines(path),
        "expected": N_TRAIN if name == "canonical_train" else N_TEST,
    })

canonical_counts_df = pd.DataFrame(canonical_counts)
canonical_counts_df

In [ ]:
def read_jsonl_sample(path: Path, n: int = 3):
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

            if len(rows) >= n:
                break

    return rows


canonical_train_sample = read_jsonl_sample(canonical_files["canonical_train"], n=3)
canonical_test_sample = read_jsonl_sample(canonical_files["canonical_test"], n=3)

pd.DataFrame(canonical_train_sample)

In [ ]:
pd.DataFrame(canonical_test_sample)

## 5. Validação do schema canônico

Cada instância canônica deve conter os campos necessários para reconstruir os cenários experimentais.

Campos esperados:

- `id`;
- `source`;
- `target_dataset`;
- `injected_dataset`;
- `task_type`;
- `injected_task_type`;
- `trusted_instruction`;
- `untrusted_data`;
- `injected_instruction`;
- `expected_answer`;
- `injected_answer`;
- `clean_prompt`;
- `attacked_prompt`;
- `attack_type`;
- `seen_attack`;
- `split`.

Esses campos permitem separar instrução legítima, dado externo, instrução maliciosa, resposta esperada, resposta adversarial e informações de split experimental.

In [ ]:
EXPECTED_CANONICAL_COLUMNS = {
    "id",
    "source",
    "target_dataset",
    "injected_dataset",
    "task_type",
    "injected_task_type",
    "trusted_instruction",
    "untrusted_data",
    "injected_instruction",
    "expected_answer",
    "injected_answer",
    "clean_prompt",
    "attacked_prompt",
    "attack_type",
    "seen_attack",
    "split",
}


def validate_jsonl_schema(path: Path, expected_columns: set, max_rows: int | None = None):
    missing_by_row = []
    extra_columns = set()
    total_rows = 0

    with path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue

            row = json.loads(line)
            row_columns = set(row.keys())

            missing = expected_columns - row_columns
            extra = row_columns - expected_columns

            if missing:
                missing_by_row.append({
                    "row_index": idx,
                    "missing": sorted(missing),
                })

            extra_columns.update(extra)
            total_rows += 1

            if max_rows is not None and total_rows >= max_rows:
                break

    return {
        "path": str(path),
        "total_rows_checked": total_rows,
        "rows_with_missing_columns": len(missing_by_row),
        "extra_columns": sorted(extra_columns),
        "valid": len(missing_by_row) == 0,
    }


schema_validation = [
    validate_jsonl_schema(canonical_files["canonical_train"], EXPECTED_CANONICAL_COLUMNS),
    validate_jsonl_schema(canonical_files["canonical_test"], EXPECTED_CANONICAL_COLUMNS),
]

pd.DataFrame(schema_validation)

## 6. Criação das views de treinamento e avaliação

A partir dos dados canônicos, são criadas views específicas para cada cenário experimental:

- `train_struq_sft.jsonl`: treinamento SFT do cenário C2;
- `train_secalign_dpo.jsonl`: treinamento DPO do cenário C3;
- `train_ih_sft.jsonl`: treinamento SFT do cenário C4;
- `test_all_scenarios.jsonl`: avaliação dos cenários C0, C1, C2, C3, C4 e entrada limpa.

Comando utilizado:

```bash
python scripts/build_training_views.py \
  2>&1 | tee logs/02_build_training_views_5k_872.
```

In [ ]:
views_log_path = LOGS_DIR / "02_build_training_views_5k_872.log"

views_command = [
    sys.executable,
    str(SCRIPTS_DIR / "build_training_views.py"),
]

print("Comando de geração das views:")
print(" ".join(views_command))

if RUN_DATA_PREP:
    result = subprocess.run(
        views_command,
        capture_output=True,
        text=True,
    )

    views_log_path.write_text(
        result.stdout + "\n" + result.stderr,
        encoding="utf-8"
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Erro ao gerar views de treinamento e avaliação.")

    print("Log salvo em:", views_log_path)
else:
    print("RUN_DATA_PREP=False. A geração das views não foi executada nesta célula.")

In [ ]:
view_files = {
    "train_struq_sft": PROCESSED_DATA_DIR / "train_struq_sft.jsonl",
    "train_secalign_dpo": PROCESSED_DATA_DIR / "train_secalign_dpo.jsonl",
    "train_ih_sft": PROCESSED_DATA_DIR / "train_ih_sft.jsonl",
    "test_all_scenarios": PROCESSED_DATA_DIR / "test_all_scenarios.jsonl",
}

view_file_checks = []

for name, path in view_files.items():
    view_file_checks.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else None,
        "lines": count_jsonl_lines(path),
    })

pd.DataFrame(view_file_checks)

## 7. Contagens esperadas

Para a configuração `n_train = 5000` e `n_test = 872`, as contagens esperadas são:

- `canonical_train.jsonl`: 5000 linhas;
- `canonical_test.jsonl`: 872 linhas;
- `train_struq_sft.jsonl`: 5000 linhas;
- `train_secalign_dpo.jsonl`: 5000 linhas;
- `train_ih_sft.jsonl`: 5000 linhas;
- `test_all_scenarios.jsonl`: 5232 linhas.

O arquivo `test_all_scenarios.jsonl` possui 5232 linhas porque cada exemplo de teste gera 6 variantes:

```text
872 exemplos × 6 variantes = 5232 linhas
```

In [ ]:
expected_line_counts = {
    "canonical_train": N_TRAIN,
    "canonical_test": N_TEST,
    "train_struq_sft": N_TRAIN,
    "train_secalign_dpo": N_TRAIN,
    "train_ih_sft": N_TRAIN,
    "test_all_scenarios": N_TEST * 6,
}

all_data_files = {}
all_data_files.update(canonical_files)
all_data_files.update(view_files)

line_count_rows = []

for name, path in all_data_files.items():
    observed = count_jsonl_lines(path)
    expected = expected_line_counts.get(name)

    line_count_rows.append({
        "file": name,
        "observed_lines": observed,
        "expected_lines": expected,
        "matches_expected": observed == expected,
        "path": str(path),
    })

line_count_df = pd.DataFrame(line_count_rows)
line_count_df

In [ ]:
expected_line_counts = {
    "canonical_train": N_TRAIN,
    "canonical_test": N_TEST,
    "train_struq_sft": N_TRAIN,
    "train_secalign_dpo": N_TRAIN,
    "train_ih_sft": N_TRAIN,
    "test_all_scenarios": N_TEST * 6,
}

all_data_files = {}
all_data_files.update(canonical_files)
all_data_files.update(view_files)

line_count_rows = []

for name, path in all_data_files.items():
    observed = count_jsonl_lines(path)
    expected = expected_line_counts.get(name)

    line_count_rows.append({
        "file": name,
        "observed_lines": observed,
        "expected_lines": expected,
        "matches_expected": observed == expected,
        "path": str(path),
    })

line_count_df = pd.DataFrame(line_count_rows)
line_count_df

## 8. Inspeção das views de treinamento

As views de treinamento têm formatos diferentes conforme o tipo de defesa.

### C2 — StruQ-like SFT

O arquivo `train_struq_sft.jsonl` contém uma coluna `text`, com o prompt estruturado e a resposta segura esperada.

### C3 — SecAlign-like DPO

O arquivo `train_secalign_dpo.jsonl` contém as colunas:

- `prompt`;
- `chosen`;
- `rejected`.

A resposta `chosen` representa o comportamento seguro esperado. A resposta `rejected` representa o comportamento induzido pela injeção.

### C4 — Instruction-Hierarchy-like SFT

O arquivo `train_ih_sft.jsonl` contém uma coluna `text`, com instruções organizadas em níveis explícitos de autoridade.

In [ ]:
struq_sample = read_jsonl_sample(view_files["train_struq_sft"], n=2)
dpo_sample = read_jsonl_sample(view_files["train_secalign_dpo"], n=2)
ih_sample = read_jsonl_sample(view_files["train_ih_sft"], n=2)

print("C2 — StruQ-like SFT")
display(pd.DataFrame(struq_sample))

print("C3 — SecAlign-like DPO")
display(pd.DataFrame(dpo_sample))

print("C4 — Instruction-Hierarchy-like SFT")
display(pd.DataFrame(ih_sample))

In [ ]:
test_sample = read_jsonl_sample(view_files["test_all_scenarios"], n=6)

pd.DataFrame(test_sample)[[
    "id",
    "scenario",
    "attack_type",
    "seen_attack",
    "is_clean",
    "expected_answer",
    "injected_answer",
]]

## 9. Distribuição dos cenários de teste

O arquivo `test_all_scenarios.jsonl` reúne as variantes de avaliação para:

- C0: modelo base sem defesa;
- C1: StruQ format-only;
- C2: StruQ-like SFT;
- C3: SecAlign-like DPO;
- C4: Instruction-Hierarchy-like SFT;
- CLEAN: avaliação limpa de utilidade.

Essa organização permite avaliar todos os cenários sobre os mesmos exemplos base.

In [ ]:
test_rows = []

with view_files["test_all_scenarios"].open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_rows.append(json.loads(line))

test_df = pd.DataFrame(test_rows)

scenario_distribution = (
    test_df
    .groupby(["scenario", "is_clean"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["scenario", "is_clean"])
)

scenario_distribution

In [ ]:
attack_distribution = (
    test_df
    .groupby(["scenario", "seen_attack", "attack_type"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["scenario", "seen_attack", "attack_type"])
)

attack_distribution

## 10. Cópia versionada dos dados da execução

Como `data/processed` funciona como área ativa de trabalho, os dados finais usados na execução são copiados para a pasta versionada da run:

```text
runs/opi_sst2_sms_5k_872/dataset_seed42/data/
```

Essa cópia garante que os dados utilizados na execução sejam preservados mesmo que data/processed seja sobrescrito posteriormente.

In [ ]:
import shutil

COPY_DATA_TO_RUN_DIR = True

if COPY_DATA_TO_RUN_DIR:
    RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)

    for name, path in all_data_files.items():
        destination = RUN_DATA_DIR / path.name

        if path.exists():
            shutil.copy2(path, destination)
            print(f"[OK] {path.name} -> {destination}")
        else:
            print(f"[WARN] Arquivo não encontrado: {path}")
else:
    print("COPY_DATA_TO_RUN_DIR=False. Nenhuma cópia foi realizada.")

In [ ]:
run_data_files = sorted(RUN_DATA_DIR.glob("*.jsonl"))

run_data_summary = pd.DataFrame([
    {
        "file": path.name,
        "lines": count_jsonl_lines(path),
        "size_mb": round(path.stat().st_size / 1024**2, 3),
        "path": str(path),
    }
    for path in run_data_files
])

run_data_summary

## 11. Manifesto dos dados

Para facilitar auditoria e reprodução, um manifesto dos dados é salvo junto da execução.

Esse manifesto registra:

- nome do experimento;
- seed do dataset;
- número de exemplos de treino;
- número de exemplos de teste;
- arquivos gerados;
- contagens de linhas;
- data/hora de criação do manifesto.

In [ ]:
data_manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_seed": DATASET_SEED,
    "n_train": N_TRAIN,
    "n_test": N_TEST,
    "created_at": datetime.now().isoformat(),
    "project_dir": str(PROJECT_DIR),
    "run_root": str(RUN_ROOT),
    "run_data_dir": str(RUN_DATA_DIR),
    "files": [
        {
            "name": path.name,
            "path": str(path),
            "lines": count_jsonl_lines(path),
            "size_mb": round(path.stat().st_size / 1024**2, 3),
        }
        for path in run_data_files
    ],
}

manifest_path = RUN_DATA_DIR / "data_manifest.json"

with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(data_manifest, f, ensure_ascii=False, indent=2)

print("Manifesto salvo em:", manifest_path)

In [ ]:
with manifest_path.open("r", encoding="utf-8") as f:
    manifest_loaded = json.load(f)

manifest_loaded